# Model 7

In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"     # agama
os.environ["MKL_NUM_THREADS"] = "1"     # numpy, scipy
os.environ["OPENBLAS_NUM_THREADS"] = "1"    # numpy
os.environ["NUMEXPR_NUM_THREADS"] = "1"     # pandas

import agama
import torch 
import numpy as np
from scipy import integrate
from astropy import units as u

from sbi.utils import BoxUniform
from sbi.inference import SNLE, simulate_for_sbi, prepare_for_sbi
from sbi.utils import likelihood_nn

from sklearn.metrics import mean_squared_error, r2_score

import pandas as pd
import pickle
import matplotlib.pyplot as plt
from galaxy_generation import generate_galaxy_multiple
from prior_generation import generate_prior


import corner

torch.set_num_threads(4)


In [2]:
# set agama unit to be in Msun, kpc, km/s
agama.setUnits(mass=1 * u.Msun, length=1*u.kpc, velocity=1 * u.km /u.s)

In [3]:
agama.setRandomSeed(13)
torch.manual_seed(13)
np.random.seed(13)


## Generation of Data Set

In [6]:

train_x_raw = np.array(pd.read_csv("training_x(poisson).csv", header=None))
train_theta_raw = np.array(pd.read_csv("training_theta(poisson).csv", header=None))

index = np.unique(train_theta_raw, axis=0, return_index=True)[1]
index = np.sort(index)


train_x_medium = np.split(train_x_raw, index, axis=0)[1:]
train_theta = train_theta_raw[index]

train_x_welldone = [x.flatten() for x in train_x_medium]

In [7]:
max_entries = 150
train_x = []
for x in train_x_welldone:
    num_pads = max_entries* 6 - x.shape[0]
    train_x.append(np.pad(x.flatten(), (0, num_pads)))
    
train_x = np.array(train_x)

Standardizing

In [8]:
theta_mean = np.mean(train_theta, axis=0)
theta_std = np.std(train_theta, axis=0)

theta_train = (train_theta - theta_mean)/ theta_std

x_mean = np.mean(train_x, axis=0)
x_std = np.std(train_x, axis=0)

x_train = (train_x - x_mean)/ x_std

/tmp/ipykernel_373124/1333393081.py:9: RuntimeWarning: invalid value encountered in divide
  x_train = (train_x - x_mean)/ x_std


In [10]:
x_train

array([[ 0.22777613, -0.28773275, -0.03634243, ...,         nan,
                nan,         nan],
       [ 0.04169299, -0.00731429,  0.00461518, ...,         nan,
                nan,         nan],
       [ 0.36612627,  0.16236392,  0.13158003, ...,         nan,
                nan,         nan],
       ...,
       [-0.00347155,  0.01619528,  0.03436687, ...,         nan,
                nan,         nan],
       [-0.00131961, -0.06539478,  0.00501244, ...,         nan,
                nan,         nan],
       [-0.04853137,  0.0259502 , -0.00287837, ...,         nan,
                nan,         nan]])

In [9]:
train_x = torch.tensor(x_train).float()
train_theta = torch.tensor(theta_train).float()

torch.set_num_threads(4)
prior_sbi = generate_prior()

# density_estimator =likelihood_nn(model="maf", 
#                                  hidden_features = 97,
#                                  num_transforms = 10,
#                                  num_bins = 10)

# inference = SNLE(prior=prior_sbi, density_estimator=density_estimator)

inference = SNLE(prior=prior_sbi)
inference.append_simulations(train_theta, train_x)
arg = {
        "training_batch_size": 512,
        "learning_rate": 0.00337562831493679,
        "validation_fraction": 0.1,
        "stop_after_epochs": 10,
        "max_num_epochs": 2**31 - 1,
        "clip_max_norm": 5.0,
        "resume_training": False,
        "discard_prior_samples": False,
        "retrain_from_scratch": False,
        "show_train_summary": True,
        # "dataloader_kwargs": {"num_workers": 2, 
        #                         "persistent_workers": True}
}

likelihood_estimator = inference.train(**arg)

# lr: 0.001347, tbs: 512

ValueError: Found 10000 NaN simulations and 0 Inf simulations.SNLE does not allow invalid simulations.Replace the invalid values with an unreasonably low or high value.

In [63]:
a = likelihood_estimator.sample(1, context=train_theta[0])